<div dir="rtl">

# FastFrames-S

مدل ساخت ویدیو از متن اپل (Apple AIML) — ساخت کلیپ ۵ ثانیه‌ای (۸۱ فریم، 832×480) روی کولب رایگان (T4).

**روش اجرا:** از منوی Runtime گزینه‌ی Run all را بزنید؛ در پایان، لینک پنل ساخته می‌شود و با وارد کردن پرامپت، ویدیو ساخته می‌شود.

**نکته‌ها:**
- توکن هاگینگ‌فیس اجباری نیست، ولی بهتر است در سکرت‌های کولب با نام HF_TOKEN بگذارید: دانلود مدل‌ها از هاگینگ‌فیس با توکن سریع‌تر است و مدل‌های نیازمند قبول لایسنس هم با همان توکن باز می‌شوند. نوت‌بوک خودش توکن را می‌خواند.
- هر بار فقط یک ویدیو بسازید؛ اجرای هم‌زمان حافظه را پر می‌کند.
- زمان تقریبی ساخت هر کلیپ روی T4 رایگان: حدود ۲ دقیقه.

کد اصلی مدل: https://github.com/apple-aiml-research/ml-fastframes

</div>

In [ ]:
import os, shutil, subprocess, sys, time

BASE = "/content/fastframes-s"
REPO = f"{BASE}/ml-fastframes"
os.makedirs(BASE, exist_ok=True)

if not os.path.exists(REPO):
    print("در حال دریافت کد مدل...", flush=True)
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/apple-aiml-research/ml-fastframes", REPO], check=True)
print("✅ کد آماده است", flush=True)

if shutil.which("uv") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)

def run_stream(cmd, cwd=None):
    p = subprocess.Popen(cmd, cwd=cwd, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end="", flush=True)
    p.wait()
    assert p.returncode == 0, f"خطا در اجرا: {cmd}"

print("در حال ساخت محیط (بار اول چند دقیقه طول می‌کشد)...", flush=True)
t0 = time.time()
run_stream(["uv", "sync"], cwd=REPO)
run_stream(["uv", "pip", "install", "--python", f"{REPO}/.venv/bin/python",
            "gradio", "huggingface_hub"], cwd=REPO)
print(f"✅ محیط آماده شد — {time.time() - t0:.0f} ثانیه", flush=True)

In [ ]:
%%writefile /content/fastframes-s/download.py
from pathlib import Path

# توکن هاگینگ‌فیس را سلول اجراکننده از سکرت کولب می‌خواند و با متغیر محیطی
# HF_TOKEN به این اسکریپت می‌رساند؛ huggingface_hub خودش آن را برمی‌دارد.

from huggingface_hub import snapshot_download

CK = Path("/content/fastframes-s/checkpoints")
CK.mkdir(parents=True, exist_ok=True)

print("دانلود ۱ از ۳: انکودر متن و توکنایزر Wan2.1-1.3B ...", flush=True)
snapshot_download("Wan-AI/Wan2.1-T2V-1.3B", local_dir=str(CK / "Wan2.1-T2V-1.3B"),
                  ignore_patterns=["diffusion_pytorch_model*"])
print("دانلود ۲ از ۳: مولد Causal-Forcing++ ...", flush=True)
snapshot_download("zhuhz22/Causal-Forcing", local_dir=str(CK),
                  allow_patterns=["causal-forcing++/framewise-2step.pt"])
print("دانلود ۳ از ۳: افزودنی‌های FastFrames-S ...", flush=True)
snapshot_download("apple/FastFrames-S", local_dir=str(CK),
                  allow_patterns=["fastframes-s/*"])

need = [CK / "Wan2.1-T2V-1.3B" / "models_t5_umt5-xxl-enc-bf16.pth",
        CK / "causal-forcing++" / "framewise-2step.pt",
        CK / "fastframes-s" / "kv_refiner.safetensors",
        CK / "fastframes-s" / "pixel_decoder.safetensors"]
total = 0.0
for p in need:
    assert p.exists(), f"فایل ناقص است: {p}"
    gb = p.stat().st_size / 2**30
    total += gb
    print(f"  {p.name}: {gb:.2f} GB", flush=True)
print(f"✅ دانلود کامل شد — مجموع فایل‌های اصلی: {total:.2f} GB", flush=True)


In [ ]:
import os, subprocess, time

env = dict(os.environ)
try:
    from google.colab import userdata
    tok = userdata.get("HF_TOKEN")
    if tok:
        env["HF_TOKEN"] = tok
        print("توکن هاگینگ‌فیس از سکرت کولب خوانده شد", flush=True)
except Exception:
    print("توکن پیدا نشد؛ دانلود بدون توکن ادامه پیدا می‌کند", flush=True)

t0 = time.time()
p = subprocess.Popen(["/content/fastframes-s/ml-fastframes/.venv/bin/python",
                      "/content/fastframes-s/download.py"],
                     env=env,
                     stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                     text=True, bufsize=1)
for line in p.stdout:
    print(line, end="", flush=True)
p.wait()
assert p.returncode == 0, "دانلود ناموفق بود"
print(f"⏱ زمان دانلود: {time.time() - t0:.0f} ثانیه", flush=True)


In [ ]:
%%writefile /content/fastframes-s/ff_encode.py
# مرحله ۱: ساخت امبدینگ متن با umT5-XXL روی GPU، در یک پروسه جدا.
# انکودر 11.36GB است و کنار بقیه مدل روی T4 جا نمی‌شود؛ این پروسه بعد از
# ذخیره امبدینگ (چند مگابایت) بسته می‌شود و VRAM کامل آزاد می‌ماند.
import argparse, hashlib, sys, time
from pathlib import Path

import torch

BASE = Path("/content/fastframes-s")
sys.path.insert(0, str(BASE / "ml-fastframes"))
from fastframes.wan.t5 import umt5_xxl_encoder
from fastframes.wan.tokenizers import HuggingfaceTokenizer


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--prompt", required=True)
    args = ap.parse_args()

    key = hashlib.sha256(args.prompt.encode()).hexdigest()[:16]
    out = BASE / "embeds" / f"emb_{key}.pt"
    out.parent.mkdir(parents=True, exist_ok=True)
    if out.exists():
        print(f"EMBEDS_CACHED {out}", flush=True)
        return

    t0 = time.perf_counter()
    wan_dir = BASE / "checkpoints" / "Wan2.1-T2V-1.3B"
    # ساخت روی متا + لود mmap + assign: بدون این، کپی‌های CPU رم کولب را پر می‌کند
    enc = umt5_xxl_encoder(dtype=torch.bfloat16, device="meta")
    state = torch.load(wan_dir / "models_t5_umt5-xxl-enc-bf16.pth",
                       map_location="cpu", weights_only=True, mmap=True)
    enc.load_state_dict(state, strict=True, assign=True)
    del state
    enc = enc.to("cuda").eval().requires_grad_(False)
    print(f"انکودر روی GPU بارگذاری شد — {time.perf_counter() - t0:.1f} ثانیه", flush=True)

    tok = HuggingfaceTokenizer(name=str(wan_dir / "google" / "umt5-xxl"),
                                seq_len=512, clean="whitespace", local_files_only=True)
    ids, mask = tok([args.prompt], return_mask=True, add_special_tokens=True)
    ids, mask = ids.to("cuda"), mask.to("cuda")
    with torch.inference_mode():
        ctx = enc(ids, mask)
        for u, n in zip(ctx, mask.gt(0).sum(dim=1).long()):
            u[n:] = 0.0
    torch.save(ctx.cpu(), out)
    print(f"EMBEDS_SAVED {out} sec={time.perf_counter() - t0:.1f} "
          f"peak_gb={torch.cuda.max_memory_allocated() / 2**30:.2f}", flush=True)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /content/fastframes-s/ff_generate.py
# مرحله ۲: تولید ویدیو در یک پروسه جدا — فقط ست تولید روی GPU می‌آید
# (DiT ۲.۸GB + ریفاینر + دیکودر + کش KV) و انکودر متن اصلاً بارگذاری نمی‌شود.
import argparse, os, sys, time
from pathlib import Path

import torch

BASE = Path("/content/fastframes-s")
sys.path.insert(0, str(BASE / "ml-fastframes"))
from fastframes.checkpoints import build_wan, load_state_dict
from fastframes.models.kv_refinement import KVRefinementNetwork
from fastframes.models.pixel_decoder import BridgedPixelDecoder, PixelDecoder
from fastframes.pipeline import MODELS, FastFrames, configure_backends
from fastframes.video import save_video
from fastframes.wan.model import WAN_CONFIGS

# کارت T4 کرنل اتنشن cuDNN ندارد و کد اپل CUDNN_ATTENTION را اجبار می‌کند؛
# بک‌اند MATH هم ماتریس کامل اتنشن را می‌سازد و VRAM تمام می‌شود. پس روی sm75
# همه فراخوانی‌های SDPA را با fp16 و بک‌اند کم‌حافظه EFFICIENT اجرا می‌کنیم
# (فقط همین پروسه؛ فایل‌های اپل دست نمی‌خورند)
import torch.nn.functional as _F
from torch.nn.attention import SDPBackend, sdpa_kernel as _sdpa_kernel

_orig_sdpa = _F.scaled_dot_product_attention


def _t4_sdpa(q, k, v, *a, **kw):
    if q.is_cuda and q.dtype == torch.bfloat16 and torch.cuda.get_device_capability()[0] < 8:
        try:
            with _sdpa_kernel([SDPBackend.EFFICIENT_ATTENTION]):
                return _orig_sdpa(q.to(torch.float16), k.to(torch.float16),
                                  v.to(torch.float16), *a, **kw).to(q.dtype)
        except RuntimeError:
            with _sdpa_kernel([SDPBackend.MATH]):
                if kw.get("is_causal"):
                    return _orig_sdpa(q, k, v, *a, **kw)
                out = []
                for i in range(0, q.shape[-2], 128):
                    out.append(_orig_sdpa(q[:, :, i:i + 128], k, v, *a, **kw))
                return torch.cat(out, dim=-2)
    return _orig_sdpa(q, k, v, *a, **kw)


_F.scaled_dot_product_attention = _t4_sdpa


def build_generator(spec):
    cfg = spec.config
    root = BASE / "checkpoints"
    dtype = torch.float16 if os.environ.get("FF_DTYPE") == "fp16" else torch.bfloat16
    dit = build_wan(cfg.wan_model, load_state_dict(root / spec.dit, "generator_ema"),
                    device="cuda", dtype=dtype)
    wan = WAN_CONFIGS[cfg.wan_model]
    refiner_state = load_state_dict(root / spec.refiner, "adapter")
    refiner = KVRefinementNetwork(
        dim=wan.dim, num_heads=wan.num_heads, head_dim=wan.dim // wan.num_heads,
        num_blocks=wan.num_layers, d_a=refiner_state["stem_in.weight"].shape[0]).to(torch.bfloat16)
    refiner.load_state_dict(refiner_state, strict=True)
    dec_state = load_state_dict(root / spec.decoder)
    if any(k.startswith("bridge.") for k in dec_state):
        dec = BridgedPixelDecoder(in_dim=wan.dim)
    else:
        dec = PixelDecoder(wan.dim)
        dec_state = {k.removeprefix("pixel_head."): v for k, v in dec_state.items()}
    dec.load_state_dict(dec_state, strict=True)
    return FastFrames(cfg, dit,
                      refiner.to("cuda").eval().requires_grad_(False),
                      dec.to("cuda", dtype=cfg.decoder_dtype).eval().requires_grad_(False),
                      None)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--embeds", required=True)
    ap.add_argument("--seed", type=int, default=0)
    ap.add_argument("--output", required=True)
    args = ap.parse_args()

    spec = MODELS["ff-s"]
    configure_backends(spec.config)
    t0 = time.perf_counter()
    model = build_generator(spec)
    t_load = time.perf_counter() - t0
    print(f"ست تولید بارگذاری شد — {t_load:.1f} ثانیه", flush=True)

    embeds = torch.load(args.embeds, map_location="cuda", weights_only=True)
    t1 = time.perf_counter()
    frames = model.generate("", args.seed, prompt_embeds=embeds)
    t_gen = time.perf_counter() - t1
    save_video(args.output, frames)
    print(f"VIDEO_SAVED {args.output} load_sec={t_load:.1f} gen_sec={t_gen:.1f} "
          f"peak_gb={torch.cuda.max_memory_allocated() / 2**30:.2f}", flush=True)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /content/fastframes-s/panel_app.py
# پنل FastFrames-S — هر تولید در دو پروسه جدا اجرا می‌شود تا VRAM تمیز بماند.
import hashlib, os, subprocess, time
from pathlib import Path

import gradio as gr

BASE = Path("/content/fastframes-s")
PY = str(BASE / "ml-fastframes" / ".venv" / "bin" / "python")
ENV = dict(os.environ, PYTORCH_CUDA_ALLOC_CONF="expandable_segments:True")


def stream(cmd, timeout, log_lines):
    p = subprocess.Popen(cmd, cwd=str(BASE), env=ENV, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1)
    t0 = time.time()
    for line in p.stdout:
        log_lines.append(line.rstrip())
        yield "\n".join(log_lines[-6:])
        if time.time() - t0 > timeout:
            p.kill()
            raise gr.Error("زمان اجرا بیش از حد طول کشید")
    p.wait()
    if p.returncode != 0:
        raise gr.Error("خطا در اجرا:\n" + "\n".join(log_lines[-6:]))


def generate(prompt, seed):
    prompt = (prompt or "").strip()
    if not prompt:
        yield "⚠️ اول پرامپت را بنویسید.", None
        return
    t0 = time.time()
    key = hashlib.sha256(prompt.encode()).hexdigest()[:16]
    emb = BASE / "embeds" / f"emb_{key}.pt"
    log = []
    yield "⏳ مرحله ۱ از ۲: آماده‌سازی متن…", None
    for tail in stream([PY, "ff_encode.py", "--prompt", prompt], 1200, log):
        yield "⏳ مرحله ۱ از ۲: آماده‌سازی متن…\n" + tail, None
    out = BASE / "outputs" / f"ff_{int(time.time())}.mp4"
    out.parent.mkdir(parents=True, exist_ok=True)
    yield "⏳ مرحله ۲ از ۲: ساخت ویدیو…", None
    for tail in stream([PY, "ff_generate.py", "--embeds", str(emb),
                        "--seed", str(int(seed)), "--output", str(out)], 2400, log):
        yield "⏳ مرحله ۲ از ۲: ساخت ویدیو…\n" + tail, None
    yield f"✅ آماده شد — مجموع {time.time() - t0:.0f} ثانیه", str(out)


with gr.Blocks(title="FastFrames-S") as demo:
    gr.Markdown("# FastFrames-S")
    prompt = gr.Textbox(label="پرامپت", lines=3,
                        value="A red fox walking through snowy pine trees toward the camera, gentle snowfall, cinematic winter light")
    seed = gr.Number(label="سید", value=0, precision=0)
    btn = gr.Button("ساخت ویدیو", variant="primary")
    status = gr.Textbox(label="وضعیت", lines=7)
    video = gr.Video(label="ویدیو")
    btn.click(generate, [prompt, seed], [status, video])

demo.queue().launch(share=True, allowed_paths=["/content/fastframes-s/outputs"])


In [ ]:
import re, subprocess, time

subprocess.run(["pkill", "-f", "panel_app.py"], capture_output=True)
time.sleep(2)
LOG = "/content/fastframes-s/panel.log"
log = open(LOG, "w")
proc = subprocess.Popen(["uv", "run", "python", "-u", "/content/fastframes-s/panel_app.py"],
                        cwd="/content/fastframes-s/ml-fastframes",
                        stdout=log, stderr=subprocess.STDOUT, start_new_session=True)
print("پنل در حال راه‌اندازی است...", flush=True)
url = None
for i in range(48):
    time.sleep(5)
    try:
        text = open(LOG).read()
    except FileNotFoundError:
        text = ""
    m = re.search(r"Running on public URL:\s*(\S+)", text)
    if m:
        url = m.group(1)
        break
    if proc.poll() is not None:
        print(text[-2000:], flush=True)
        raise RuntimeError("پنل بالا نیامد — لاگ بالا را ببینید")
    print(f"⏳ در حال راه‌اندازی… {(i + 1) * 5} ثانیه", flush=True)
print(f"✅ لینک پنل: {url}", flush=True)

<div dir="rtl">

**مشخصات خروجی:** ویدیوی MP4 با ابعاد 832×480 و ۸۱ فریم (حدود ۵ ثانیه).

**درباره‌ی مدل:** FastFrames-S نسخه‌ی سبک و سریع خانواده‌ی FastFrames اپل است و برای اجرا روی یک GPU معمولی طراحی شده. وزن‌ها و کد از مخزن رسمی اپل و هاگینگ‌فیس گرفته می‌شوند؛ برای شرایط استفاده، لایسنس همان مخزن‌ها را ببینید.

</div>